1. Write SQL statements to create:
- A catalog named telecom_catalog_assign
- A schema landing_zone
- A volume landing_vol
- Using dbutils.fs.mkdirs, create folders:
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/ 
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/ 
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
- Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):
 a. Volume vs DBFS/FileStore
 b. Why production teams prefer Volumes for regulated data

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS telecom_catalog_assign;

show catalogs;



In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;
show schemas in telecom_catalog_assign;


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;

show volumes in telecom_catalog_assign.landing_zone;

In [0]:
folders = [
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/",
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/",
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/"
]

for folder in folders:
    dbutils.fs.mkdirs(folder)

display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/"))

###Data files to use in this usecase:
customer_csv = ''' 101,Arun,31,Chennai,PREPAID 102,Meera,45,Bangalore,POSTPAID 103,Irfan,29,Hyderabad,PREPAID 104,Raj,52,Mumbai,POSTPAID 105,,27,Delhi,PREPAID 106,Sneha,abc,Pune,PREPAID '''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count 101\t320\t1500\t20 102\t120\t4000\t5 103\t540\t600\t52 104\t45\t200\t2 105\t0\t0\t0 '''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp 5001|101|TWR01|-80|2025-01-10 10:21:54 5004|104|TWR05|-75|2025-01-10 11:01:12 '''

###2. Filesystem operations
- Write dbutils.fs.put code to copy the above datasets into your created Volume folders:
Customer data → /Volumes/.../customer/
Usage data → /Volumes/.../usage/
Tower data (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/
- Write a command to validate whether files were successfully copied

In [0]:
customer_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/"
usage_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/"
tower_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/"

customer_csv = """customer_id,name,age,city,plan_type
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
"""

usage_tsv = """customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
"""

tower_logs_region1 = """event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
"""

tower_logs_region2 = """event_id|customer_id|tower_id|signal_strength|timestamp
5002|102|TWR08|-82|2025-01-10 10:35:10
5003|103|TWR09|-68|2025-01-10 10:48:05
"""

dbutils.fs.put(f"{customer_path}customer.csv", customer_csv, overwrite=True)
dbutils.fs.put(f"{usage_path}usage.tsv", usage_tsv, overwrite=True)
dbutils.fs.put(f"{tower_path}region1/tower_logs.psv", tower_logs_region1, overwrite=True)
dbutils.fs.put(f"{tower_path}region2/tower_logs.psv", tower_logs_region2, overwrite=True)

In [0]:
target_paths = [
    customer_path,
    usage_path,
    f"{tower_path}region1/",
    f"{tower_path}region2/",
]

for path in target_paths:
    print(f"=== {path} ===")
    for file_info in dbutils.fs.ls(path):
        print(f"  {file_info.name} ({file_info.size} bytes) -> {file_info.path}")
        if file_info.size > 0:
            print(dbutils.fs.head(file_info.path, max_bytes=1024))

##Spark Directory Read Use Cases
Read all tower logs using: Path glob filter (example: *.csv) Multiple paths input Recursive lookup

Demonstrate these 3 reads separately: Using pathGlobFilter Using list of paths in spark.read.csv([path1, path2]) Using .option("recursiveFileLookup","true")

Compare the outputs and understand when each should be used.

In [0]:
tower_base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"

df_glob = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "|")
    .option("inferSchema", "true")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.psv")  # change to "*.csv" if reading CSV files
    .load(tower_base)
)

display(df_glob)

In [0]:
tower_paths = [
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs.psv",
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/tower_logs.psv",
]

df_multi = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "|")
    .option("inferSchema", "true")
    .load(tower_paths)
)

display(df_multi)

In [0]:
tower_base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"

df_recursive = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "|")
    .option("inferSchema", "true")
    .option("recursiveFileLookup", "true")
    .load(tower_base)
)

display(df_recursive)

###4. Schema Inference, Header, and Separator
- Try the Customer, Usage files with the option and options using read.csv and format function:
header=false, inferSchema=false
or
header=true, inferSchema=true
- Write a note on What changed when we use header or inferSchema with true/false?
- How schema inference handled “abc” in age?

In [0]:
customer_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/"

df_cust_f_opt = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .csv(customer_path)
)

display(df_cust_f_opt)
df_cust_f_opt.printSchema()

In [0]:
df_cust_t_opts = (
    spark.read
    .format("csv")
    .options(**{"header": "true", "inferSchema": "true"})
    .load(customer_path)
)

display(df_cust_t_opts)
df_cust_t_opts.printSchema()

Toggling header between true and false controls whether the first line is used as column names (customer_id, voice_mins, etc.) and stripped from the data or left inside as a normal data row with positional _c0, _c1 names, while toggling inferSchema controls whether Spark scans data rows to guess types (int, etc.) versus keeping every column as string (with header=false keeping the header text in row 1 often forcing columns to stay string even when inferSchema=true).

When inferSchema=true encounters "abc" in the age column of customer.csv, Spark's type inference sees a non-numeric string alongside numbers (31, 45, 29, 52, 27) and widens/falls back the inferred type of age to string (StringType) rather than failing the read or turning "abc" into null during inference—preserving the raw value so you can detect, cast, or quarantine bad records downstream.

###Column Renaming Usecases
- Apply column names using string using toDF function for customer data
- Apply column names and datatype using the schema function for usage data
- Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data

In [0]:
#Apply column names using string using toDF function for customer data
customer_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/"

df_cust_raw = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .csv(customer_path)
)

df_cust_named = (
    df_cust_raw
    .toDF("customer_id", "name", "age", "city", "plan_type")
    .filter("customer_id != 'customer_id'")  # remove header row if read with header=false
)

display(df_cust_named)
df_cust_named.printSchema()

In [0]:
from pyspark.sql.types import IntegerType, StructField, StructType

usage_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/"

usage_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("voice_mins", IntegerType(), True),
    StructField("data_mb", IntegerType(), True),
    StructField("sms_count", IntegerType(), True),
])

#Apply column names and datatype using the schema function for usage data
df_usage = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .schema(usage_schema)
    .csv(usage_path)
)

display(df_usage)
df_usage.printSchema()

In [0]:
#Alternative DDL Schema string in .schema(..)
usage_ddl_schema = "customer_id INT, voice_mins INT, data_mb INT, sms_count INT"

df_usage_ddl = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .schema(usage_ddl_schema)
    .csv(usage_path)
)

display(df_usage_ddl)
df_usage_ddl.printSchema()

In [0]:
#Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

tower_schema = StructType([
    StructField("event_id", IntegerType(), nullable=True),
    StructField("customer_id", IntegerType(), nullable=True),
    StructField("tower_id", StringType(), nullable=True),
    StructField("signal_strength", IntegerType(), nullable=True),
    StructField("timestamp", TimestampType(), nullable=True),
])

5.1 Spark Write Operations using
csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options

In [0]:
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

tower_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"

tower_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("tower_id", StringType(), True),
    StructField("signal_strength", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
])

df_tower = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "|")
    .option("recursiveFileLookup", "true")
    .schema(tower_schema)
    .load(tower_path)
)

# Verify df_tower is loaded
display(df_tower)